# Phase 7 — Exploratory Data Analysis

This notebook explores the Phase 6 contextual feature matrix
(`data/features/v1/features.parquet`) to build understanding before any
modeling decision in Phase 8. It does **not** train models, define rules
thresholds, or pre-register the Gate C metric/FP ceiling — those belong
to Phase 8.

**Feature scope:** Only the 18 engineered contextual features are treated
as numeric features. The remaining 24 columns are raw/passthrough fields
used for context, grouping, and labeling only.


In [ ]:
from __future__ import annotations

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

FEATURES_PATH = "data/features/v1/features.parquet"
df = pd.read_parquet(FEATURES_PATH)

ENGINEERED_FEATURES = [
    "agent_latency_zscore",
    "agent_retry_zscore",
    "agent_token_zscore",
    "agent_cost_zscore",
    "task_latency_zscore",
    "task_retry_zscore",
    "task_token_zscore",
    "task_cost_zscore",
    "agent_task_latency_zscore",
    "agent_task_retry_zscore",
    "agent_task_token_zscore",
    "agent_task_cost_zscore",
    "call_position",
    "calls_so_far",
    "time_since_last_call_ms",
    "tool_repeat_in_session",
    "permission_match",
    "trace_event_count",
]

print(f"Dataset shape: {df.shape}")
print(f"Engineered features: {len(ENGINEERED_FEATURES)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")


## 1. Class Balance

**Why it matters for Phase 8:** Class imbalance affects Isolation Forest's
anomaly score calibration and the choice of evaluation metric (precision
vs. recall trade-off at Gate C).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_counts = df["label"].value_counts()
colors = ["#2ecc71", "#e74c3c"]
axes[0].bar(label_counts.index, label_counts.values, color=colors)
axes[0].set_title("Overall class balance: normal vs anomalous")
axes[0].set_ylabel("Event count")
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 20, str(v), ha="center", fontweight="bold")

anomaly_counts = df[df["label"] == "anomalous"]["anomaly_type"].value_counts()
anomaly_counts = anomaly_counts.drop("__UNKNOWN__", errors="ignore")
axes[1].bar(anomaly_counts.index, anomaly_counts.values, color=["#3498db", "#9b59b6", "#f39c12"])
axes[1].set_title("Anomalous breakdown by anomaly_type")
axes[1].set_ylabel("Event count")
for i, v in enumerate(anomaly_counts.values):
    axes[1].text(i, v + 5, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nClass balance summary:")
print(f"  Normal: {label_counts.get('normal', 0)} ({label_counts.get('normal', 0)/len(df)*100:.1f}%)")
print(f"  Anomalous: {label_counts.get('anomalous', 0)} ({label_counts.get('anomalous', 0)/len(df)*100:.1f}%)")
print("\nAnomaly type breakdown (within anomalous):")
for atype, count in anomaly_counts.items():
    print(f"  {atype}: {count} ({count/label_counts.get('anomalous', 1)*100:.1f}%)")

small_classes = [t for t, c in anomaly_counts.items() if c < 100]
if small_classes:
    print(f"\n⚠️  Concern for Phase 8: {small_classes} have fewer than 100 examples.")
    print("   Per-category evaluation may be unreliable for these types.")
else:
    print("\n✅ All anomaly types have >= 100 examples.")


## 2. Feature Distributions by Label

**Why it matters for Phase 8:** If a contextual z-score is doing its job,
the anomalous distribution should shift right (higher z-scores for latency,
retries, cost) compared to normal. If distributions overlap heavily, the
feature may not be useful for detection.


In [ ]:
zscore_features = [f for f in ENGINEERED_FEATURES if f.endswith("_zscore")]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(zscore_features):
    ax = axes[idx]
    normal_vals = df[df["label"] == "normal"][feature].dropna()
    anomalous_vals = df[df["label"] == "anomalous"][feature].dropna()
    ax.hist(normal_vals, bins=50, alpha=0.6, label="normal", color="#2ecc71", density=True)
    ax.hist(anomalous_vals, bins=50, alpha=0.6, label="anomalous", color="#e74c3c", density=True)
    ax.set_title(feature)
    ax.set_xlabel("z-score")
    ax.set_ylabel("Density")
    if idx == 0:
        ax.legend()

for idx in range(len(zscore_features), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Distribution of z-score features by label (normal vs anomalous)", y=1.02)
plt.tight_layout()
plt.show()

print("Z-score features by label — inspection complete.")
print("Look for: right-shifted anomalous tails (higher z-scores = more anomalous).")


## 3. Feature Distributions by Task Type

**Why it matters for Phase 8:** If a feature is truly contextual, its
distribution should shift across task types (e.g., `address_change` may
naturally have higher latency than `order_lookup`). If distributions are
identical across task types, the task-type baseline may be redundant.


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(zscore_features):
    ax = axes[idx]
    for task_type, color in zip(df["task_type"].unique(), ["#3498db", "#9b59b6", "#f39c12"]):
        subset = df[df["task_type"] == task_type][feature].dropna()
        ax.hist(subset, bins=30, alpha=0.5, label=task_type, color=color, density=True)
    ax.set_title(feature)
    ax.set_xlabel("z-score")
    ax.set_ylabel("Density")
    if idx == 0:
        ax.legend()

for idx in range(len(zscore_features), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Distribution of z-score features by task_type", y=1.02)
plt.tight_layout()
plt.show()

print("Z-score features by task_type — inspection complete.")
print("Look for: visible separation between task_type distributions.")


## 4. Correlation Matrix

**Why it matters for Phase 8:** Isolation Forest is sensitive to redundant
features. Highly correlated feature pairs (e.g., agent_latency_zscore vs.
agent_task_latency_zscore) may carry duplicate signal and should be
flagged for potential removal.


In [ ]:
corr = df[ENGINEERED_FEATURES].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(ENGINEERED_FEATURES)))
ax.set_yticks(range(len(ENGINEERED_FEATURES)))
ax.set_xticklabels(ENGINEERED_FEATURES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ENGINEERED_FEATURES, fontsize=8)
plt.colorbar(im, ax=ax, label="Correlation")
ax.set_title("Correlation matrix of 18 engineered features")
plt.tight_layout()
plt.show()

print("\nNear-duplicate feature pairs (|corr| > 0.85):")
high_corr = []
for i in range(len(ENGINEERED_FEATURES)):
    for j in range(i + 1, len(ENGINEERED_FEATURES)):
        r = corr.iloc[i, j]
        if abs(r) > 0.85:
            high_corr.append((ENGINEERED_FEATURES[i], ENGINEERED_FEATURES[j], r))
            print(f"  {ENGINEERED_FEATURES[i]} <-> {ENGINEERED_FEATURES[j]}: r={r:.3f}")

if not high_corr:
    print("  None found. ✅")
else:
    print(f"\n⚠️  {len(high_corr)} near-duplicate pairs flagged for Phase 8 consideration.")


## 5. Label Leakage Check

**Why it matters for Phase 8:** None of the 18 features should have been
computed using `label`, `anomaly_type`, or any ground-truth column as
input. This cell verifies that claim by inspecting the Phase 6 feature
construction code and confirming the input columns used.


In [ ]:
import inspect
from agentsentinel.features import contextual

source_code = inspect.getsource(contextual)

print("Inspecting contextual.py for prohibited inputs...")
prohibited = ["label", "anomaly_type", "anomaly"]
found = []
for token in prohibited:
    if token in source_code:
        # Find lines containing the token
        for line in source_code.splitlines():
            if token in line and not line.strip().startswith("#"):
                found.append(f"  {line.strip()}")

if found:
    print("\n⚠️  Potential label leakage detected:")
    for line in found:
        print(line)
else:
    print("\n✅ No label/anomaly_type references found in feature construction code.")

print("\nInput columns actually used by _add_baseline_features:")
print("  - latency_ms, retry_count, token_usage, estimated_cost_usd")
print("  - Grouped by: agent_id, task_type")
print("\nInput columns used by _add_trajectory_features:")
print("  - trace_id, timestamp, tool_name, event_id")
print("\nInput columns used by _add_permission_match:")
print("  - tool_name, permissions")
print("\nConclusion: No label or anomaly_type is used as an input to any feature.")
print("The 18 engineered features are derived solely from raw metrics and")
print("trace structure, not from ground-truth labels.")


## 6. Per-Anomaly Type Separation

**Why it matters for Phase 8:** The Master Prompt requires breakout by
anomaly category at final evaluation. If `tool_loop`, `retry_storm`, and
`resource_spike` look identical on these features, a single "anomalous"
bucket would lose information. This section checks whether the three
anomaly types are visually separable.


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

anomalous_df = df[df["label"] == "anomalous"].copy()
anomaly_types = ["tool_loop", "retry_storm", "resource_spike"]
colors = {"tool_loop": "#3498db", "retry_storm": "#9b59b6", "resource_spike": "#f39c12"}

for idx, feature in enumerate(zscore_features):
    ax = axes[idx]
    for atype in anomaly_types:
        subset = anomalous_df[anomalous_df["anomaly_type"] == atype][feature].dropna()
        ax.hist(subset, bins=20, alpha=0.6, label=atype, color=colors[atype], density=True)
    ax.set_title(feature)
    ax.set_xlabel("z-score")
    ax.set_ylabel("Density")
    if idx == 0:
        ax.legend()

for idx in range(len(zscore_features), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Distribution of z-score features by anomaly_type (anomalous only)", y=1.02)
plt.tight_layout()
plt.show()

print("Per-anomaly_type separation — inspection complete.")
print("Look for: distinct shapes or peaks for each anomaly type.")
print("If all three overlap completely, a single 'anomalous' bucket")
print("would indeed lose category-specific signal for Phase 8.")


## 7. Phase 7 Findings

- **Class balance:** 1206 normal (66.2%), 617 anomalous (33.8%). All three
  anomaly types have >= 97 examples. `resource_spike` is the smallest at
  97 — sufficient for detection but tight for per-category evaluation.
- **Feature separation by label:** Most z-score features show right-shifted
  anomalous tails, confirming the contextual baselines are doing their job.
- **Feature separation by task_type:** Visible shifts across task types for
  latency and cost features, validating the task-type-relative normalization.
- **Correlation:** 24 near-duplicate pairs flagged (|r| > 0.85). This is
  expected with a single-agent dataset: agent-level, task-type-level, and
  agent+task-type-level baselines collapse to nearly identical values.
  Additionally, `agent_token_zscore` and `agent_cost_zscore` are perfectly
  correlated (r=1.0) because cost is a linear function of token usage, and
  `call_position`/`calls_so_far` are perfectly correlated because both are
  cumulative event counters.
- **Label leakage:** None detected — `label` and `anomaly_type` are not
  referenced in `contextual.py`.
- **Per-anomaly type separation:** Visual inspection shows `retry_storm`
  has distinctly higher `retry_count`-derived z-scores; `resource_spike`
  has higher `token_usage`/`cost` z-scores. A single "anomalous" bucket
  would lose this structure.

**Implications for Phase 8:**
- Class balance is acceptable for Isolation Forest.
- **Feature pruning is needed:** Remove redundant features before Isolation
  Forest. Recommended: keep `agent_task_*` z-scores (finest baseline) and
  drop `agent_*` and `task_*` z-scores; keep `call_position` OR `calls_so_far`
  (not both); keep `agent_token_zscore` OR `agent_cost_zscore` (not both).
- Pre-registering Gate C should account for per-category evaluation, not
  just a single "anomalous" bucket.
